# Clustering Jerárquico Aglomerativo (HAC)
## Dataset: Hotel Bookings

El **Clustering Jerárquico Aglomerativo** (HAC) agrupa observaciones similares construyendo
una jerarquía ascendente: cada observación comienza como su propio cluster y se van fusionando
los más parecidos hasta obtener un único árbol llamado **dendrograma**.

La clase `HAC` hereda de `NoSupervisado → AnalisisDatosExploratorio`, por lo que
expone todos los métodos de limpieza y transformación del EDA directamente sobre el objeto.

### Flujo de trabajo
```
mf.HAC(path, num)          # 1. Cargar datos
    ↓ métodos EDA heredados # 2. Limpiar y preparar
hac.ajustar()              # 3. Ajustar el modelo
hac.plot_*()               # 4. Visualizar resultados
```

## 1. Importación de librerías

In [ ]:
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
import pckEDA as mf

%matplotlib inline
print('pckEDA cargado correctamente')
print('Clase HAC disponible:', mf.HAC)

## 2. Carga de datos

`HAC` hereda de `AnalisisDatosExploratorio`, por lo que carga el CSV directamente.
- Modo `1`: separador coma, primera columna como índice.

In [ ]:
hac = mf.HAC('../Semana 8/hotel_bookings.csv', 1, n_clusters=4, metodo='ward')

hac.mostrarTamaño()
hac.muestraTiposDeDatos()

## 3. Preprocesamiento con métodos heredados del EDA

### 3.1 Codificación de variables categóricas

In [ ]:
hac.codificarCategorica('hotel')
hac.codificarCategorica('arrival_date_month')
hac.codificarCategorica('assigned_room_type')
hac.codificarCategorica('deposit_type', mapeo={'No Deposit': 0, 'Non Refund': 1, 'Refundable': 2})
hac.codificarCategorica('customer_type')
hac.codificarCategorica('reservation_status')

### 3.2 Limpieza de nulos y duplicados

In [ ]:
hac.eliminarDuplicados()
hac.eliminarNulos()

### 3.3 Conservar solo columnas numéricas

In [ ]:
hac.analisisNumerico()
print(f'Columnas para el modelo: {list(hac.df.columns)}')
print(f'Observaciones: {len(hac.df)}')

## 4. Ajuste del modelo HAC

Con los datos ya limpios, se llama a `ajustar()` para entrenar el modelo.

In [ ]:
hac.ajustar()

print(f'Método de enlace  : {hac.metodo}')
print(f'Número de clusters: {hac.n_clusters}')
print(f'Correlación cofenética: {hac.cophenet_corr:.4f}')
print()
import pandas as pd
print('Distribución de observaciones por cluster:')
print(pd.Series(hac.etiquetas).value_counts().sort_index())

> **Correlación cofenética ≥ 0.70** → clustering aceptable. **≥ 0.80** → muy buena calidad.

## 5. Visualizaciones

### 5.1 Dendrograma

In [ ]:
plt.figure(figsize=(14, 6))
hac.plot_dendrograma(max_hojas=50, titulo='Dendrograma HAC — Hotel Bookings (Ward)')
plt.tight_layout()
plt.show()

### 5.2 Mapa de calor — Perfil de clusters

In [ ]:
plt.figure(figsize=(16, 5))
hac.plot_mapa_calor(titulo='Perfil de Clusters — Hotel Bookings')
plt.tight_layout()
plt.show()

### 5.3 Distribución de observaciones por cluster

In [ ]:
plt.figure(figsize=(8, 5))
hac.plot_distribucion(titulo='Distribución de Observaciones por Cluster')
plt.tight_layout()
plt.show()

### 5.4 Diagramas de dispersión por cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
hac.plot_dispersion('adr', 'lead_time', titulo='Tarifa Diaria vs Anticipación de Reserva')

plt.sca(axes[1])
hac.plot_dispersion('adr', 'stays_in_week_nights', titulo='Tarifa Diaria vs Noches entre Semana')

plt.tight_layout()
plt.show()

## 6. Resumen estadístico por cluster

In [ ]:
hac.resumen.style.background_gradient(cmap='Blues', axis=0).format('{:.2f}')

## 7. Comparación de métodos de enlace

In [ ]:
metodos = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, metodo in zip(axes.ravel(), metodos):
    h = mf.HAC('../Semana 8/hotel_bookings.csv', 1, n_clusters=4, metodo=metodo)
    h.codificarCategorica('hotel')
    h.codificarCategorica('arrival_date_month')
    h.codificarCategorica('assigned_room_type')
    h.codificarCategorica('deposit_type', mapeo={'No Deposit': 0, 'Non Refund': 1, 'Refundable': 2})
    h.codificarCategorica('customer_type')
    h.codificarCategorica('reservation_status')
    h.eliminarNulos()
    h.analisisNumerico()
    h.ajustar()
    plt.sca(ax)
    h.plot_dendrograma(
        max_hojas=30,
        titulo=f'Enlace: {metodo.capitalize()} — Cofenética: {h.cophenet_corr:.3f}'
    )

plt.suptitle('Comparación de Métodos de Enlace — HAC', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Conclusiones

- El método **Ward** produce clusters equilibrados y obtiene la mejor correlación cofenética en datos tabulares.
- El **mapa de calor** revela los perfiles de cada cluster: grupos con alta tarifa (`adr`), largo tiempo de anticipación (`lead_time`), etc.
- Los **diagramas de dispersión** muestran qué tan separados están los clusters en el espacio de variables originales.
- La **correlación cofenética** es el indicador de calidad principal: cuanto más cercana a 1, mejor preserva la jerarquía las distancias reales.